In [1]:
## This is a notebook dedicated to introducing the 'pydeck' package

## We will use this dataset ---> https://gis-txdot.opendata.arcgis.com/datasets/TXDOT::txdot-roadway-inventory/about
## from TXDOT for the VMT of everyroadway in the state.



## Import some packages!

In [2]:
import pandas as pd
import geopandas as gpd ## my personal favorite python package <<<3333 plz respect the goat geopandas (and the amazing packages it is built on top of, i.e. leaflet)
import numpy as np

from shapely.geometry import box

import h3pandas

import pydeck

import osmnx

pd.set_option('display.max_columns', None) ### This line makes all the columns display, rather than ellipses shorten

## Import the data!

In [3]:
#texas_road_inventory = gpd.read_file("C:/Users/aaron/Fun Mapping Work/Data/Houston_hexes_data/TxDOT_Roadway_Inventory_3771128114195576813.gpkg")
houston_area_road_inventory = gpd.read_file("C:/Users/aaron/Fun Mapping Work/Houston_Hexes/Houston_Area_road_VMT.gpkg")

In [4]:
houston_area_road_inventory.head()

,NUM_LANES,LN_MILES,DVMT,geometry
0,2,0.398,45.770,"MULTILINESTRING ((-95.30826 29.519, -95.30825 ..."
1,2,0.316,36.340,"MULTILINESTRING ((-95.31385 29.52083, -95.3127..."
2,2,1.684,491.728,"MULTILINESTRING ((-95.31381 29.51821, -95.3138..."
3,2,0.322,37.030,"MULTILINESTRING ((-95.31387 29.52218, -95.3112..."
4,2,1.212,139.380,"MULTILINESTRING ((-95.30847 29.52863, -95.3084..."


In [5]:
#type(texas_road_inventory)
#texas_road_inventory_trimmed = texas_road_inventory[['NUM_LANES', 'LN_MILES', 'DVMT', 'geometry']] ## only the columns we need, this line goes a long way with processing speed.
#texas_road_inventory_trimmed = texas_road_inventory_trimmed.to_crs('4326')

## get the city boundary of houston

In [6]:
#houston_boundary = osmnx.geocode_to_gdf('Houston, United States of America')

In [7]:
#houston_boundary.explore()

## decide that the actual city boundary is inferior to the BBOX here. 

In [8]:
## greater houston bounding box. Shoutout bboxfinder.com
#houston_bbox = [-95.718384,29.518210,-95.028992,29.965569]
#houston_polygon = box(*houston_bbox) ## this line turns the coordinates into a bounding box using the shapely package.
#houston_bbox_gdf = gpd.GeoDataFrame(geometry=[houston_polygon], crs = '4326')
#houston_roads_gdf = texas_road_inventory_trimmed.clip(houston_bbox_gdf)

In [9]:
#houston_roads_gdf.to_file('Houston_Area_road_VMT.gpkg', driver='GPKG', layer='name')
#houston_roads_gdf.to_file('Houston_Area_road_VMT.geojson', driver='GeoJSON')
#houston_roads_gdf.to_file('Houston_Area_road_VMT.shp')

In [10]:
#houston_roads_gdf.explore()

In [11]:
#houston_bbox_gdf.explore()

In [12]:
#houston_roads_gdf.explore()

In [13]:
#houston_roads_gdf.head(2)

## Create both centroid and lat / long columns (prep for different visualizations)

In [14]:
## create centroid column
houston_area_road_inventory['geometry'] = houston_area_road_inventory['geometry'].centroid
houston_area_road_inventory['lat'] = houston_area_road_inventory['geometry'].y
houston_area_road_inventory['long'] = houston_area_road_inventory['geometry'].x

C:\Users\aaron\AppData\Local\Temp\ipykernel_43588\3560065042.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  houston_area_road_inventory['geometry'] = houston_area_road_inventory['geometry'].centroid


In [15]:
houston_area_road_inventory.head()

,NUM_LANES,LN_MILES,DVMT,geometry,lat,long
0,2,0.398,45.770,POINT (-95.30826 29.51861),29.518606,-95.308255
1,2,0.316,36.340,POINT (-95.31254 29.52085),29.520845,-95.312538
2,2,1.684,491.728,POINT (-95.3139 29.52324),29.523240,-95.313896
3,2,0.322,37.030,POINT (-95.31254 29.5222),29.522197,-95.312536
4,2,1.212,139.380,POINT (-95.30836 29.52422),29.524224,-95.308363


In [16]:
pydeck_data = houston_area_road_inventory[['DVMT', 'lat', 'long']] ## this one is to make the PyDeck 'hexagonlayer'
centroid_frame = houston_area_road_inventory[['DVMT', 'geometry']] ## this one is for the H3 visualization

In [17]:
#centroid_frame.head()

## H3 Visualization

In [18]:
hexagon_frame = centroid_frame.h3.geo_to_h3_aggregate(8) ## invoking H3 in one simple line!

In [19]:
#hexMap = hexagon_frame.explore(column = 'DVMT', vmax = 100000) ## declare the map

In [20]:
#hexMap ## render the map

In [21]:
#hexMap.save('Houston_Hexagons_GPD.html')  ## save the map to your machine

## Pydeck Hex visualization

In [22]:
## check out different PyDeck options here ---> https://deckgl.readthedocs.io/en/latest/#gallery

In [23]:
## hexagon (vertical) layer


layer = pydeck.Layer(
    "HexagonLayer",
    pydeck_data,
    get_position=["long", "lat"],
    radius = 1000, ## changes the size of the hexes, important!

    get_elevation_weight = 'DVMT', ##determines the height
    get_color_weight = 'DVMT', ##determines the color

    opacity = 0.4,
    auto_highlight=True,
    elevation_scale=50,
    pickable=True,
    elevation_range=[0, 3000],
    extruded=True,
    coverage=1,
)

In [24]:
## determines the starting view 
view_state = pydeck.ViewState(
    longitude=-95.415,
    latitude=29.6,
    zoom=8,
    min_zoom=5,
    max_zoom=15,
    pitch=0, ## adjusts the viewing height
    bearing=0, ## adjusts where the 'camera' is looking on from, in the E/W direction
)


In [25]:
pdk_hex_map_render = pydeck.Deck(layers=[layer], initial_view_state=view_state) ## declares the map

In [26]:
#pdk_hex_map_render

In [27]:
#map_render.to_html("houston_pydeck_hexagon_layer.html") ## SAVES THE FILE

## Attempt to make PyDeck Heatmap Layer

In [28]:
## code largely taken from official pydeck tutorial --> https://deckgl.readthedocs.io/en/latest/gallery/heatmap_layer.html

In [29]:
COLOR_BREWER_BLUE_SCALE = [
    [240, 249, 232],
    [204, 235, 197],
    [168, 221, 181],
    [123, 204, 196],
    [67, 162, 202],
    [8, 104, 172],
]

In [30]:
vmt_heat = pydeck.Layer(
    "HeatmapLayer",
    data=pydeck_data,
    opacity=0.9,
    get_position=["long", "lat"],
    #aggregation=pydeck.types.String("MEAN"),
    color_range=COLOR_BREWER_BLUE_SCALE,
    threshold=1,
    get_weight="DVMT",
    pickable=True,
)

In [31]:
heatmap_render = pydeck.Deck(
    layers=[vmt_heat], ## can easily include multiple layers here
    initial_view_state=view_state,
    tooltip={"text": "Daily Vehicle Miles Traveled"},
)

In [32]:
#heatmap_render